# boIA — prueba de CowDB de Ganader-IA

Este cuaderno incorpora el dataset público `ganader-ia-developers/cowdb` como **experimento auxiliar de estimación de peso por imagen**. No reemplaza el detector aéreo YOLO11 de boIA: La ficha de CowDB no acredita que sus imágenes sean tomas aéreas de dron, así que no se asume que represente el dominio de boIA. La prueba separa animales completos entre train/valid/test para que fotos de una misma vaca no inflen las métricas.

El dataset no muestra una licencia clara en su ficha. Usarlo para esta evaluación privada no implica permiso para redistribuir imágenes, pesos entrenados o derivados; confirmar condiciones con quien lo publicó antes de compartir o comercializar resultados.


## 1. Entorno y carpetas

En Colab, seleccioná un runtime GPU (por ejemplo T4) para ejecutar la regresión visual. La inspección de datos y el baseline de mediana funcionan también en CPU.

In [ ]:
%pip -q install datasets scikit-learn

from pathlib import Path
import os, random, json, math
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED)
BOIA_ROOT = Path('/content/boIA')

DATA_DIR = BOIA_ROOT / 'datasets' / 'ganaderia'
RESULTS_DIR = BOIA_ROOT / 'results' / 'ganaderia_cowdb_weight_baseline'
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('boIA:', BOIA_ROOT)
print('GPU disponible:', __import__('torch').cuda.is_available())

## 2. Descargar CowDB e inspeccionar etiquetas

Se carga la versión `resized` (876 ejemplos; alrededor de 871 MB). El repositorio también ofrece una variante más pesada. Se conserva `cow_id` solo para hacer la partición por animal; **no se le da al modelo como entrada**.

In [ ]:
REPO_ID = 'ganader-ia-developers/cowdb'
data = load_dataset(REPO_ID, 'resized', split='train', cache_dir=str(DATA_DIR / 'hf_cache'))
print(data)
print('Columnas:', data.column_names)
print('Features:', data.features)

required = {'image', 'cow_id', 'weight'}
missing = required - set(data.column_names)
if missing:
    raise ValueError(f'Faltan columnas esperadas en CowDB: {missing}')

# Metadata liviana: no lleva las imágenes a RAM.
meta = data.remove_columns(['image']).to_pandas().reset_index(drop=True)
meta['weight'] = pd.to_numeric(meta['weight'], errors='coerce')
meta['cow_id'] = pd.to_numeric(meta['cow_id'], errors='coerce')
meta = meta.dropna(subset=['weight', 'cow_id']).copy()
meta['cow_id'] = meta['cow_id'].astype(int)
meta['row_idx'] = meta.index.astype(int)
print('Filas utilizables:', len(meta))
print('Animales:', meta['cow_id'].nunique())
print('Rango de peso (kg):', meta['weight'].min(), 'a', meta['weight'].max())
display(meta.drop(columns='row_idx').describe(include='all').T)
display(meta.groupby('cow_id').agg(n_imagenes=('weight','size'), peso=('weight','first')).head(12))
print('Peso repetido dentro de cada animal:', (meta.groupby('cow_id')['weight'].nunique().max() == 1))

In [ ]:
# Distribución de pesos y ejemplos visuales para detectar orientación/calidad.
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
meta['weight'].hist(bins=25, ax=ax[0]); ax[0].set_title('Peso etiquetado'); ax[0].set_xlabel('kg')
meta.groupby('cow_id').size().hist(bins=20, ax=ax[1]); ax[1].set_title('Imágenes por animal'); ax[1].set_xlabel('imágenes')
plt.tight_layout(); plt.show()

sample_idx = meta['row_idx'].sample(min(12, len(meta)), random_state=SEED).tolist()
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for ax, idx in zip(axes.flat, sample_idx):
    row = data[int(idx)]
    ax.imshow(row['image'].convert('RGB'))
    ax.set_title(f"cow_id={row['cow_id']} · {row['weight']} kg\n{row.get('orientation','')}", fontsize=8)
    ax.axis('off')
for ax in axes.flat[len(sample_idx):]: ax.axis('off')
plt.tight_layout(); plt.show()

## 3. Partición por animal y baseline

Las fotos de cada `cow_id` quedan enteras en un solo subconjunto. La mediana del peso de train da una referencia mínima: el modelo visual tiene que superarla en test para justificar más trabajo.

In [ ]:
animal_ids = np.array(sorted(meta['cow_id'].unique()))
trainval_ids, test_ids = train_test_split(animal_ids, test_size=0.20, random_state=SEED)
train_ids, val_ids = train_test_split(trainval_ids, test_size=0.20, random_state=SEED)
splits = {
    'train': meta.loc[meta.cow_id.isin(train_ids), 'row_idx'].tolist(),
    'val': meta.loc[meta.cow_id.isin(val_ids), 'row_idx'].tolist(),
    'test': meta.loc[meta.cow_id.isin(test_ids), 'row_idx'].tolist(),
}
id_splits = {'train': set(train_ids), 'val': set(val_ids), 'test': set(test_ids)}
assert not (id_splits['train'] & id_splits['val'] or id_splits['train'] & id_splits['test'] or id_splits['val'] & id_splits['test'])
for name, ids in id_splits.items(): print(name, 'animales=', len(ids), 'imágenes=', len(splits[name]))

# Baseline constante: siempre predice la mediana de peso del conjunto de entrenamiento.
y_train = meta.set_index('row_idx').loc[splits['train'], 'weight'].to_numpy(float)
y_test = meta.set_index('row_idx').loc[splits['test'], 'weight'].to_numpy(float)
dummy = DummyRegressor(strategy='median').fit(np.zeros((len(y_train), 1)), y_train)
pred_dummy = dummy.predict(np.zeros((len(y_test), 1)))
def score_weights(y_true, y_pred, label):
    return {'modelo': label, 'MAE_kg': mean_absolute_error(y_true, y_pred),
            'RMSE_kg': mean_squared_error(y_true, y_pred) ** 0.5,
            'sesgo_kg': float(np.mean(y_pred - y_true)),
            'MAPE_pct': float(np.mean(np.abs((y_pred-y_true)/np.maximum(np.abs(y_true), 1e-6))) * 100),
            'R2': r2_score(y_true, y_pred)}
baseline = score_weights(y_test, pred_dummy, 'mediana_train')
display(pd.DataFrame([baseline]))

## 4. Regresión visual exploratoria

Entrena un MobileNetV3-Small preentrenado, con la base congelada y una cabeza de regresión liviana. Se ejecuta solo si Colab ofrece GPU; no se modifica ni se mezcla con el detector YOLO11 de vistas aéreas. Se selecciona por MAE de validación y se reporta una sola vez en test.

In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import MobileNet_V3_Small_Weights

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dispositivo:', device)

if device.type != 'cuda':
    print('Salteo el entrenamiento de imágenes: activá GPU en Entorno de ejecución → Cambiar tipo de entorno de ejecución.')
    print('La carga, inspección y baseline ya se ejecutaron; no hay un resultado CNN para reportar.')
else:
    weights = MobileNet_V3_Small_Weights.DEFAULT
    train_tf = transforms.Compose([transforms.Resize(256), transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
                                   transforms.RandomHorizontalFlip(), transforms.ToTensor(),
                                   transforms.Normalize(weights.transforms().mean, weights.transforms().std)])
    eval_tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
                                  transforms.Normalize(weights.transforms().mean, weights.transforms().std)])
    target_mean, target_std = float(y_train.mean()), float(y_train.std() or 1.0)
    class CowWeightDataset(Dataset):
        def __init__(self, row_indices, transform): self.row_indices, self.transform = row_indices, transform
        def __len__(self): return len(self.row_indices)
        def __getitem__(self, j):
            row = data[int(self.row_indices[j])]
            image = row['image'].convert('RGB')
            weight = (float(row['weight']) - target_mean) / target_std
            return self.transform(image), torch.tensor(weight, dtype=torch.float32)
    loaders = {
        'train': DataLoader(CowWeightDataset(splits['train'], train_tf), batch_size=24, shuffle=True, num_workers=2, pin_memory=True),
        'val': DataLoader(CowWeightDataset(splits['val'], eval_tf), batch_size=24, shuffle=False, num_workers=2, pin_memory=True),
        'test': DataLoader(CowWeightDataset(splits['test'], eval_tf), batch_size=24, shuffle=False, num_workers=2, pin_memory=True),
    }
    model = models.mobilenet_v3_small(weights=weights)
    for p in model.features.parameters(): p.requires_grad = False
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 1)
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_fn = nn.SmoothL1Loss()
    best_val, best_state, history = float('inf'), None, []
    for epoch in range(1, 9):
        model.train(); train_losses=[]
        for xb, yb in loaders['train']:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True); loss=loss_fn(model(xb).squeeze(1), yb); loss.backward(); optimizer.step()
            train_losses.append(float(loss.detach().cpu()))
        model.eval(); val_losses=[]; val_true=[]; val_pred=[]
        with torch.no_grad():
            for xb, yb in loaders['val']:
                out=model(xb.to(device, non_blocking=True)).squeeze(1).cpu().numpy() * target_std + target_mean
                val_pred.extend(out.tolist()); val_true.extend((yb.numpy() * target_std + target_mean).tolist())
                val_losses.append(float(loss_fn(torch.tensor((out-target_mean)/target_std), yb).item()))
        val_mae=mean_absolute_error(val_true, val_pred)
        history.append({'epoch':epoch, 'train_smoothL1':np.mean(train_losses), 'val_MAE_kg':val_mae})
        print(f"epoch {epoch:02d} · train loss {np.mean(train_losses):.3f} · val MAE {val_mae:.1f} kg")
        if val_mae < best_val:
            best_val=val_mae; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(best_state); model.eval()
    test_true=[]; test_pred=[]
    with torch.no_grad():
        for xb, yb in loaders['test']:
            out=model(xb.to(device, non_blocking=True)).squeeze(1).cpu().numpy() * target_std + target_mean
            test_pred.extend(out.tolist()); test_true.extend((yb.numpy() * target_std + target_mean).tolist())
    image_result=score_weights(np.array(test_true), np.array(test_pred), 'MobileNetV3 imagen')
    results=pd.DataFrame([baseline, image_result]); display(results)
    pd.DataFrame(history).to_csv(RESULTS_DIR / 'training_history.csv', index=False)
    results.to_csv(RESULTS_DIR / 'test_metrics.csv', index=False)
    torch.save({'state_dict':best_state, 'target_mean':target_mean, 'target_std':target_std,
                'architecture':'mobilenet_v3_small', 'dataset':REPO_ID, 'split_seed':SEED},
               RESULTS_DIR / 'cowdb_weight_baseline.pt')
    print('Resultados guardados en:', RESULTS_DIR)

## 5. Decisión de integración en boIA

Interpretación prevista:
- Si el modelo visual no supera la mediana en animales no vistos, CowDB no da una señal útil para peso y se descarta esa ruta.
- Si la supera, el resultado es solo una prueba con imágenes de este dataset. No se traslada esa métrica a vacas filmadas desde un dron.
- Para conectar con el flujo aéreo de boIA, hace falta un conjunto de imágenes tomadas con el protocolo/dron objetivo y pesos reales asociados a cada animal.
- Mantener detector/conteo aéreo YOLO11 separado de esta regresión de imágenes hasta medirla en datos compatibles.

No se informa ninguna precisión esperada antes de ejecutar el cuaderno. El resultado que importa es MAE/RMSE/sesgo en test agrupado por `cow_id`, junto con la mediana de referencia.